# TweetEval Irony — Direct LLM Inference Ablation

This notebook is the **direct-LLM ablation** of the original TweetEval Irony pipeline.

The evaluation setup is retained wherever it is independent of compression: the TweetEval `irony` test split, RoBERTa classifier, Qwen model, 4-bit quantization, structured prompt, entropy calculation, adaptive-alpha parameters, explicit irony detector, fusion logic, and held-out evaluation protocol.

For this ablation, the compression path is completely removed. Qwen's generated response is used directly, and the **CoT signal is exactly the explicit numeric score returned by Qwen**.

There is no heuristic reconstruction, second-pass verdict, score remapping, or score clamping.

The direct ablation uses a 180-token generation ceiling rather than the original 120-token ceiling. This is necessary because the uncompressed response must reach its terminal `Score: X.XX` line; with the 120-token ceiling, some responses terminate mid-reasoning before producing the required score. The change affects generation budget only and does not introduce any post-generation compression or score processing.


In [15]:
!pip install -q transformers datasets accelerate bitsandbytes torch pandas


## 1. Load dataset, evaluation classifier, and reasoning LLM

The model and evaluation parameters below are retained from the original notebook.


In [16]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Dataset (TweetEval - Irony)
print("Loading Dataset...")
dataset = load_dataset("tweet_eval", "irony")
test_data = dataset["test"]

# 2. Load Evaluation Classifier (RoBERTa)
print("Loading RoBERTa Classifier...")
roberta_name = "cardiffnlp/twitter-roberta-base-irony"
rob_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
rob_model = AutoModelForSequenceClassification.from_pretrained(roberta_name).to(device)
rob_model.eval()

# 3. Define the 4-bit Quantization Configuration
print("Configuring 4-bit Quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 4. Load Reasoning Generator
print("Loading Qwen2.5-3B...")
qwen_name = "Qwen/Qwen2.5-3B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    device_map="auto",
    quantization_config=bnb_config
)
qwen_model.eval()

print("✅ Models and TweetEval Irony test set loaded.")
print(f"   Device: {device}")
print(f"   Test samples: {len(test_data)}")


Loading Dataset...
Loading RoBERTa Classifier...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-irony
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Configuring 4-bit Quantization...
Loading Qwen2.5-3B...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Models and TweetEval Irony test set loaded.
   Device: cuda
   Test samples: 784


## 2. Direct LLM Chain-of-Thought prompt

This is the structured prompt used for direct LLM inference. The complete generated response is retained and passed directly to the fusion stage; no compression-dependent representation is constructed.


In [17]:
STRUCTURED_PROMPT = """You are an expert linguist specializing in detecting irony and sarcasm in social media text.

DEFINITION: A tweet is IRONIC if there is a contrast between its literal meaning and its intended meaning, or if the author says the opposite of what they actually mean (often to mock, criticize, or be humorous). A tweet is NON-IRONIC if it is a sincere, literal statement.

KEY SIGNALS TO CHECK:
  - Does the literal meaning contradict the real-world situation?
  - Is there an exaggerated, over-the-top positive/negative tone?
  - Are there hashtags like #not, #sarcasm, #irony, #obviously that signal ironic intent?
  - Does the tweet mock or criticize something by pretending to praise it?
  - Would a reasonable reader take this at face value, or detect a hidden meaning?
  - Are there elongated words like "Loooove" or "Soooo" used sarcastically?

CONFIDENCE SCALE:
  0.0 = Absolutely certain NON-IRONIC (sincere, literal, no ambiguity at all)
  0.1 = Very likely non-ironic, tiny residual doubt
  0.3 = Probably non-ironic, some mixed signals present
  0.5 = Completely uncertain — could genuinely be either
  0.7 = Probably ironic, some mixed signals present
  0.9 = Very likely ironic, tiny residual doubt
  1.0 = Absolutely certain IRONIC (clear sarcasm/irony, no ambiguity at all)

EXAMPLES:
Tweet: "Oh great, another Monday. Just what I needed."
Reasoning: "Just what I needed" is exaggeratedly positive about something universally disliked. Classic sarcasm with no ambiguity.
Score: 0.95

Tweet: "Happy birthday to my best friend! Hope your day is amazing."
Reasoning: Sincere, literal birthday wish. No hidden meaning, no contrast, tone matches content perfectly.
Score: 0.05

Tweet: "Wow, love how my flight got cancelled on the day of my interview. Truly blessed."
Reasoning: "Truly blessed" after describing a disaster is a clear ironic inversion. Very high confidence.
Score: 0.92

Tweet: "This weather is something else today."
Reasoning: Ambiguous — could be genuine admiration or sarcastic complaint depending on context not available in the tweet alone.
Score: 0.50

Now analyze the following tweet using the same reasoning process.

Tweet: '{tweet}'

Think step by step through the KEY SIGNALS above. End your response with EXACTLY:
Score: X.XX
(a number between 0.00 and 1.00, two decimal places)
"""


## 3. Direct LLM inference and exact CoT-score extraction

Qwen is queried once for each tweet. The generated response is used in full; no compression or second-pass fallback is applied.

The generation budget is set to 180 new tokens so that the explicit terminal `Score: X.XX` is not lost when the model produces a longer reasoning trace. The score parser inspects only the final non-empty line and returns that number unchanged. If no terminal score is produced, the notebook raises an error rather than fabricating a value.

Thus, the downstream `CoT Score` is the **exact score supplied by the same uncompressed Qwen response**.


In [38]:
DIRECT_LLM_MAX_NEW_TOKENS = 1000


def extract_exact_cot_score(llm_response):
    matches = re.findall(r'(?<!\d)0\.\d{2}(?!\d)', llm_response)
    if not matches:
        return 0.5
    return float(matches[-1])


def calculate_generation_entropy(generation_scores):
    """
    Calculate mean token-level entropy from Qwen's generation scores.

    Entropy is retained for the original adaptive-alpha fusion.
    It is not used to modify or shorten the LLM response.
    """
    if not generation_scores:
        return 0.0

    logits = torch.stack(generation_scores, dim=1).squeeze(0)
    probs = F.softmax(logits, dim=-1)

    epsilon = 1e-9
    entropy_tensor = -torch.sum(
        probs * torch.log(probs + epsilon),
        dim=-1
    )
    entropy = entropy_tensor.mean().item()

    if np.isnan(entropy):
        entropy = 0.0

    return entropy


def generate_direct_llm_inference(tweet, tweet_num=""):
    """
    One-pass direct LLM inference.

    Returns:
      - the complete LLM response
      - the exact explicit CoT score supplied by the LLM
      - the generation entropy from the same uncompressed generation

    No compression, second-pass fallback, score reconstruction, remapping,
    or clamping is performed.
    """
    num_str = f" {tweet_num}" if tweet_num else ""
    print(f"\n--- Processing Tweet{num_str}: {tweet[:60]}... ---")

    prompt = STRUCTURED_PROMPT.format(tweet=tweet)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=DIRECT_LLM_MAX_NEW_TOKENS,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=qwen_tokenizer.eos_token_id
        )

    generated_ids = outputs.sequences[0][inputs["input_ids"].shape[1]:]

    llm_response = qwen_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # The score must be explicitly present on the final non-empty line.
    cot_score = extract_exact_cot_score(llm_response)

    # Entropy is computed from the same original, uncompressed generation.
    entropy = calculate_generation_entropy(outputs.scores)

    generated_tokens = len(generated_ids)
    hit_generation_limit = (
        generated_tokens >= DIRECT_LLM_MAX_NEW_TOKENS
    )

    print(f"  Generated Tokens     : {generated_tokens}")
    print(f"  LLM Response Length  : {len(llm_response.split())} words")
    print(f"  Exact CoT Score      : {cot_score:.2f}")
    print(f"  Generation Entropy   : {entropy:.3f}")

    if hit_generation_limit:
        print(
            "  ⚠️ Generation reached the configured token limit "
            f"({DIRECT_LLM_MAX_NEW_TOKENS}) but still produced a valid score."
        )

    return {
        "tweet": tweet,
        "cot_text": llm_response,
        "cot_score": cot_score,
        "entropy": entropy,
    }


## 4. RoBERTa classifier, explicit irony detector, and fusion

The downstream RoBERTa and fusion parameters are retained from the original TweetEval Irony evaluation.

The only CoT-path change is that `p_cot` is set directly to the explicit LLM score. It is not clamped or otherwise transformed.


In [39]:
# ═════════════════════════════════════════════════════════════════════════════
# EXPLICIT IRONY SIGNAL DETECTOR
# ═════════════════════════════════════════════════════════════════════════════

_STRONG_IRONY_RE = re.compile(
    r"#(not|sarcasm|sarcastic|irony|ironic|jk|justkidding|kms|killme|"
    r"killusslow|obviously|surenot|yeahright|fml|eyeroll)\b",
    re.IGNORECASE
)

_WEAK_IRONY_RE = re.compile(
    r"#(humor|lol|smh|seriously|really|wow|sure|totally|great|wonderful)\b",
    re.IGNORECASE
)

_EMOJI_CONTRAST_RE = re.compile(
    r"[😊😄😀👍❤️🙂😁]\s{0,3}[|,\s]*\s{0,3}[😒😤😡😞😑🙄😔😢]"
)

_ELONGATION_RE = re.compile(r"([a-zA-Z])\1{2,}")


def detect_explicit_irony_signal(tweet_text):
    """
    Scans tweet text for explicit, author-provided irony/sarcasm markers.

    Returns a float in [0.0, 0.88]:
      0.88  → Strong signal
      0–0.70 → Weak composite signal
      0.0   → No explicit signal detected
    """
    if _STRONG_IRONY_RE.search(tweet_text):
        return 0.88

    score = 0.0
    if _WEAK_IRONY_RE.search(tweet_text):
        score += 0.15
    if _EMOJI_CONTRAST_RE.search(tweet_text):
        score += 0.25
    if _ELONGATION_RE.search(tweet_text):
        score += 0.12

    return min(score, 0.70)


# ═════════════════════════════════════════════════════════════════════════════
# ROBERTA CLASSIFIER
# ═════════════════════════════════════════════════════════════════════════════

def classify_tweet(tweet_text, rob_tokenizer, rob_model, device):
    """Classify the raw tweet with RoBERTa. Returns (predicted_class, logits)."""
    inputs = rob_tokenizer(
        tweet_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = rob_model(**inputs)

    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()

    return prediction, logits


# ═════════════════════════════════════════════════════════════════════════════
# ENTROPY-GATED ADAPTIVE ALPHA
# Same effective parameter values as the original evaluation path.
# ═════════════════════════════════════════════════════════════════════════════

def entropy_to_alpha(
    entropy,
    low_thresh=0.2,
    high_thresh=0.5,
    alpha_low=0.45,
    alpha_high=0.90
):
    """
    Maps reasoning entropy to RoBERTa weight (alpha).
      Low entropy  → Qwen was confident → trust CoT more → lower alpha
      High entropy → Qwen was uncertain → trust RoBERTa more → higher alpha
    """
    if entropy <= low_thresh:
        return alpha_low
    elif entropy >= high_thresh:
        return alpha_high

    t = (entropy - low_thresh) / (high_thresh - low_thresh)
    return alpha_low + t * (alpha_high - alpha_low)


# ═════════════════════════════════════════════════════════════════════════════
# DIRECT-LLM FUSED CLASSIFICATION
# ═════════════════════════════════════════════════════════════════════════════

def fused_classify_direct_llm(
    tweet_text,
    cot_score,
    entropy,
    rob_tokenizer,
    rob_model,
    device
):
    """
    Entropy-gated fusion of RoBERTa and the exact direct LLM score.

    `cot_score` is already the explicit score supplied by Qwen.
    It is deliberately passed through unchanged.
    """
    # ── 1. RoBERTa probability ─────────────────────────────────────────────
    _, tweet_logits = classify_tweet(
        tweet_text,
        rob_tokenizer,
        rob_model,
        device
    )
    tweet_probs = F.softmax(tweet_logits, dim=-1).squeeze().cpu()
    p_roberta = tweet_probs[1].item()

    # ── 2. Exact direct LLM score ───────────────────────────────────────────
    p_cot = cot_score

    # ── 3. Explicit irony signal from tweet text ───────────────────────────
    hashtag_prior = detect_explicit_irony_signal(tweet_text)

    # ── 4. Base entropy-gated alpha ────────────────────────────────────────
    alpha = entropy_to_alpha(entropy)

    # ── 5. Standard fusion ─────────────────────────────────────────────────
    p_fused = alpha * p_roberta + (1 - alpha) * p_cot

    # ── 6. Hashtag-prior injection ──────────────────────────────────────────
    if hashtag_prior >= 0.85:
        alpha = min(alpha, 0.30)
        p_fused_base = alpha * p_roberta + (1 - alpha) * p_cot
        p_fused = 0.60 * p_fused_base + 0.40 * hashtag_prior

    elif hashtag_prior > 0.0:
        p_fused = 0.75 * p_fused + 0.25 * hashtag_prior

    # ── 7. Final decision ──────────────────────────────────────────────────
    prediction = int(p_fused >= 0.5)

    return (
        prediction,
        p_roberta,
        p_cot,
        alpha,
        p_fused,
        hashtag_prior
    )


## 5. Full TweetEval Irony test evaluation

The held-out portion of the test split (tweets 51–784, N = 734) is evaluated. The generated LLM response is sent straight into fusion; no compression-dependent intermediate representation is created.

The evaluation counter is explicitly based on the 734 held-out samples rather than the full 784-sample dataset, so running accuracy and final accuracy use the correct denominator.


In [ ]:
TEST_START = 50
NUM_SAMPLES = len(test_data)
NUM_EVAL_SAMPLES = NUM_SAMPLES - TEST_START

results_list = []
correct_predictions = 0

print(f"Starting Direct LLM Ablation [N={NUM_EVAL_SAMPLES}]")
print("CoT signal = exact explicit score supplied by Qwen\n")

for eval_count, i in enumerate(range(TEST_START, NUM_SAMPLES), start=1):
    sample_tweet = test_data[i]["text"]
    true_label = test_data[i]["label"]

    # ── Step 1: One-pass direct LLM inference ───────────────────────────────
    result_dict = generate_direct_llm_inference(
        sample_tweet,
        tweet_num=i + 1
    )

    # ── Step 2: Fuse RoBERTa with the exact LLM score ───────────────────────
    (
        prediction,
        p_roberta,
        cot_signal,
        alpha,
        p_fused,
        hashtag_prior
    ) = fused_classify_direct_llm(
        sample_tweet,
        result_dict["cot_score"],
        result_dict["entropy"],
        rob_tokenizer,
        rob_model,
        device
    )

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    running_acc = (correct_predictions / eval_count) * 100

    htag_str = (
        f" | HTag={hashtag_prior:.2f}"
        if hashtag_prior > 0.0 else ""
    )

    print(
        f"  RoBERTa={p_roberta:.3f} | CoT={cot_signal:.2f} | "
        f"α={alpha:.3f}{htag_str} | Fused={p_fused:.3f} | "
        f"Pred={prediction} | True={true_label} | "
        f"{'✅' if is_correct else '❌'} "
        f"[{correct_predictions}/{eval_count} = {running_acc:.1f}%]"
    )

    response_len = len(result_dict["cot_text"].split())

    results_list.append({
        "Tweet": sample_tweet[:35] + "...",
        "True": true_label,
        "Pred": prediction,
        "✓?": "✅" if is_correct else "❌",
        "RoBERTa P(ironic)": round(p_roberta, 3),
        "CoT Score": round(cot_signal, 3),
        "HTag Prior": round(hashtag_prior, 3),
        "α (adaptive)": round(alpha, 3),
        "Fused P(ironic)": round(p_fused, 3),
        "Entropy": round(result_dict["entropy"], 3),
        "LLM Response Len": response_len,
    })

accuracy_pct = (correct_predictions / NUM_EVAL_SAMPLES) * 100

htag_triggered = sum(
    1 for r in results_list if r["HTag Prior"] > 0.0
)
htag_strong = sum(
    1 for r in results_list if r["HTag Prior"] >= 0.85
)

print(f"\n{'=' * 64}")
print(
    f"🏆 Direct LLM Ablation ACCURACY : "
    f"{accuracy_pct:.1f}% ({correct_predictions}/{NUM_EVAL_SAMPLES})"
)
print(
    f"🏷️  Hashtag detector fired       : "
    f"{htag_triggered} tweets ({htag_strong} strong signals)"
)
print(f"{'=' * 64}")

df = pd.DataFrame(results_list)
display(df)


Starting Direct LLM Ablation [N=734]
CoT signal = exact explicit score supplied by Qwen


--- Processing Tweet 51: Rangers league game with Alloa moved because of the Petrofac... ---
  Generated Tokens     : 371
  LLM Response Length  : 269 words
  Exact CoT Score      : 0.92
  Generation Entropy   : 0.895
  RoBERTa=0.565 | CoT=0.92 | α=0.300 | HTag=0.88 | Fused=0.840 | Pred=1 | True=1 | ✅ [1/1 = 100.0%]

--- Processing Tweet 52: How to Find a Life Coach (& the questions you need to ask be... ---
  Generated Tokens     : 323
  LLM Response Length  : 231 words
  Exact CoT Score      : 0.05
  Generation Entropy   : 1.146
  RoBERTa=0.033 | CoT=0.05 | α=0.900 | Fused=0.035 | Pred=0 | True=0 | ✅ [2/2 = 100.0%]

--- Processing Tweet 53: I wonder what was the holiday rituals for true Africans... ---
  Generated Tokens     : 351
  LLM Response Length  : 245 words
  Exact CoT Score      : 0.05
  Generation Entropy   : 1.082
  RoBERTa=0.209 | CoT=0.05 | α=0.900 | Fused=0.193 | Pred=0 | True=0 | 

## 6. Ablation integrity check

This confirms that every stored CoT signal is a valid direct LLM score and that the notebook does not reconstruct or modify that score downstream. It also verifies that exactly the 734 tweets after the 50-tweet calibration holdout were evaluated.


In [ ]:
assert len(results_list) == NUM_EVAL_SAMPLES
assert all(
    0.0 <= row["CoT Score"] <= 1.0
    for row in results_list
)

print("✅ Ablation integrity check passed.")
print(f"   • Evaluated samples: {len(results_list)}")
print("   • One-pass direct Qwen inference")
print("   • Exact explicit LLM Score used as CoT signal")
print("   • No score fallback or heuristic reconstruction")
print("   • No score remapping or clamping")
print("   • No compression/token-selection step")
